# Week 5 Live Coding: Power and Null Results

Three tasks:
1. Simulate one fake experiment with a true effect
2. Compute power: what fraction of fake experiments detect the effect?
3. Vary the sample size and watch power change


### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

## Part 0: The pilot data

First, let's look at the actual canvassing pilot and confirm the field director's numbers.


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk05_power_and_nulls/data/canvassing_pilot.csv')
df.shape

In [ ]:
# The ATE
treated_mean = df[df['treatment'] == 1]['turned_out'].mean()
control_mean = df[df['treatment'] == 0]['turned_out'].mean()
observed_ate = treated_mean - control_mean
print(f'Control turnout: {control_mean:.3f}')
print(f'Treatment turnout: {treated_mean:.3f}')
print(f'ATE: {observed_ate:+.4f}')

The ATE is positive but small. Let's run randomization inference to get the p-value.


In [ ]:
# RI on the pilot data — same .sample(frac=1).values shuffle pattern as W3 and W4.
np.random.seed(2026)
fake_ates = []
for i in range(1000):
    shuffled = df['treatment'].sample(frac=1).values  # shuffle labels, keep outcomes fixed
    fake_ate = df['turned_out'][shuffled == 1].mean() - df['turned_out'][shuffled == 0].mean()
    fake_ates.append(fake_ate)

p_value = np.mean([abs(f) >= abs(observed_ate) for f in fake_ates])
print(f'p-value: {p_value:.3f}')

The p-value is large. The field director would call this "no significant effect."

But does that mean canvassing doesn't work? Or was the experiment just too small to tell?


## Part 1: Simulate one fake experiment with a true effect

Let's imagine a world where canvassing truly boosts turnout by 2 percentage points. We will simulate one experiment in that world and see whether the experiment detects the effect.

This is the key shift from Weeks 3-4: we are generating data where the treatment *actually works*. The question is whether the experiment is big enough to find it.


**Step 1: Generate the fake data.**


In [ ]:
# One fake experiment where the true effect is +2 pp
np.random.seed(2026)

n_per_group = 200
true_control_rate = 0.42
true_effect = 0.02  # +2 percentage points
true_treatment_rate = true_control_rate + true_effect

# Generate outcomes: each voter is a coin flip
control_outcomes = np.random.binomial(1, true_control_rate, size=n_per_group)
treatment_outcomes = np.random.binomial(1, true_treatment_rate, size=n_per_group)

# Compute the ATE
fake_ate = treatment_outcomes.mean() - control_outcomes.mean()
print(f'True effect: +{true_effect:.3f}')
print(f'Observed ATE in this fake experiment: {fake_ate:+.4f}')

The observed ATE may be close to the true effect, or far from it. With only 200 per group, there is a lot of noise.

**Step 2: Is this fake experiment significant?**

We check significance the same way we always do: randomization inference. Shuffle the treatment labels, recompute the ATE, repeat 1,000 times.


In [ ]:
# Check significance by RI.
# Shuffle the combined outcomes to check significance — same logic as
# .sample(frac=1) from the RI loops in Weeks 3-4.
# np.random.permutation shuffles the list, then we split it back into
# two groups and recompute the difference.
all_outcomes = np.concatenate([control_outcomes, treatment_outcomes])
n_total = len(all_outcomes)

np.random.seed(2026)
ri_ates = []
for i in range(1000):
    shuffled = np.random.permutation(all_outcomes)
    ri_ate = shuffled[:n_per_group].mean() - shuffled[n_per_group:].mean()
    ri_ates.append(ri_ate)

ri_p = np.mean([abs(f) >= abs(fake_ate) for f in ri_ates])
print(f'ATE: {fake_ate:+.4f}')
print(f'p-value: {ri_p:.3f}')
print(f'Significant at p < 0.05? {ri_p < 0.05}')

Sometimes significant, sometimes not. With only 200 per group, it depends on the luck of the draw. That is the problem.


## Part 2: Compute power

One fake experiment is not enough. We need to ask: out of 1,000 fake experiments where canvassing truly works (+2 pp), how many produce $p < 0.05$?

That fraction is the **power** of the experiment.


**The loop.** For each fake experiment, we:
1. Generate treatment and control outcomes (with a true +2 pp effect)
2. Run a quick RI check (we'll use a shortcut: 200 shuffles instead of 1,000 to keep it fast)
3. Record whether $p < 0.05$


In [ ]:
# What this loop does in plain English:
# 1. Generate one fake experiment where canvassing truly works (+2 pp)
# 2. Run a quick randomization inference (200 shuffles) to get a p-value
# 3. Record whether the p-value is below 0.05
# 4. Repeat 1,000 times
# Power = fraction of experiments that detected the effect

np.random.seed(2026)
n_per_group = 200
true_control_rate = 0.42
true_treatment_rate = 0.44  # true effect = +2 pp

detected = 0

for i in range(1000):
    if i % 200 == 0:
        print(f'Experiment {i}/1000...')
    
    # Generate one fake experiment
    control = np.random.binomial(1, true_control_rate, size=n_per_group)
    treatment = np.random.binomial(1, true_treatment_rate, size=n_per_group)
    ate = treatment.mean() - control.mean()
    
    # Quick RI: shuffle the combined outcomes 200 times — same logic as
    # the .sample(frac=1) shuffles from Weeks 3-4.
    all_outcomes = np.concatenate([control, treatment])
    ri_count = 0
    for j in range(200):
        shuffled = np.random.permutation(all_outcomes)
        ri_ate = shuffled[:n_per_group].mean() - shuffled[n_per_group:].mean()
        if abs(ri_ate) >= abs(ate):
            ri_count += 1
    p = ri_count / 200
    
    if p < 0.05:
        detected += 1

power = detected / 1000
print(f'\nSample size: {n_per_group * 2}')
print(f'True effect: +0.02')
print(f'Experiments that detected the effect: {detected} / 1000')
print(f'Power: {power:.3f}')

**Stop and read that number.** Power is around 6%. That means: even though canvassing truly works in every one of these fake experiments, the experiment only detects it about 6% of the time.

The field director's pilot had essentially no chance of finding the effect. Her null result is uninformative.


## Part 2.5: The bounce has a name — standard error and confidence interval

We keep simulating the same thing: the estimate bouncing from experiment to experiment. The typical size of that bounce has a name — the **standard error** — and it leads directly to the single most useful summary of an experiment, the **confidence interval**.

In [ ]:
# Run the experiment 2,000 times (true effect +2 pp) and look at how much the estimate bounces.
np.random.seed(2026)
n_per_group = 200
ates = []
for i in range(2000):
    control = np.random.binomial(1, 0.42, n_per_group)
    treatment = np.random.binomial(1, 0.44, n_per_group)   # true effect +2 pp
    ates.append(treatment.mean() - control.mean())
ates = np.array(ates)

print(f'Average estimate across 2,000 experiments: {100*ates.mean():+.2f} pp  (near the true +2)')
print(f'Standard deviation of the estimate        = {100*ates.std():.2f} pp  <- this is the STANDARD ERROR')

That standard deviation — about **5 percentage points** — is the **standard error (SE)**: the typical distance between a single experiment's estimate and the truth. It is the bounce, as one number.

(There's also a shortcut formula — the `sqrt(...)` you'll see in Part 3 — but its meaning is exactly this: the size of the bounce. The effect we're hunting for is +1 to +3 pp, *smaller than the bounce*. That is the whole problem.)

### The pilot's confidence interval

A **95% confidence interval** is the estimate plus or minus about 2 standard errors — the range of true effects the data cannot rule out. Compute it for the actual pilot:

In [ ]:
p_c = df[df['treatment'] == 0]['turned_out'].mean()
p_t = df[df['treatment'] == 1]['turned_out'].mean()
ate = p_t - p_c
se  = np.sqrt(p_t*(1-p_t)/200 + p_c*(1-p_c)/200)   # standard error of the difference
lo, hi = ate - 1.96*se, ate + 1.96*se

print(f'Pilot ATE:                {100*ate:+.1f} pp')
print(f'Standard error:           {100*se:.1f} pp')
print(f'95% confidence interval: [{100*lo:+.1f}, {100*hi:+.1f}] pp')

Read that interval: about **[-8, +11] percentage points**. It includes **0** (so we can't rule out that canvassing does nothing) *and* **+10** (so we can't rule out that it works enormously). The interval is so wide it's consistent with almost any conclusion.

**That width is the uninformative null** — a wide confidence interval and low power are two symptoms of the same cause: a standard error that is large relative to the effect.

*(What the "95%" means: across many repeat experiments, about 95% of the intervals you'd build this way would contain the true effect. It's a property of the method, not a 95% chance that this one interval contains the truth.)*

### Now you can read the whole regression table

Back in Weeks 2 and 3 we read only the `coef` column of a regression and skipped the rest. Run the pilot as a regression and look at the full table — every column is now something you know.

In [ ]:
smf.ols('turned_out ~ treatment', data=df).fit().summary()

Find the **`treatment`** row and read across it:

- **`coef` = 0.0150** — the ATE, +1.5 pp.
- **`std err` = 0.049** — the standard error, \~5 pp, the bounce.
- **`[0.025  0.975]` = [-0.082, 0.112]** — the 95% confidence interval.
- **`P>|t|` = 0.762** — the p-value. "Not significant" — and you can see *why*: **the confidence interval includes zero.**

Those are the exact columns we grayed out in Week 2 and skipped in Week 3. You can now read all of them.

## Part 3: Vary the sample size

What happens if we run a bigger experiment? Let's compute power at several sample sizes.

For speed, we will use a shortcut: instead of running a full RI (200 shuffles) inside each fake experiment, we use a simple rule that gives approximately the same answer for large samples. The rule: the ATE is "significant" if it is larger than about 2 standard errors (the bounce we just named in Part 2.5) — a threshold based on the sample size. The number 1.96 in the threshold corresponds to p = 0.05 — it comes from the normal distribution, which RI converges to for large samples. You do not need to memorize this; the point is that RI and this shortcut agree.

**For the curious:** Where does 1.96 come from? When the sample is large, the distribution of fake ATEs from randomization inference looks like a bell curve (a *normal distribution*). The number 1.96 marks the point where 2.5% of the bell curve falls in each tail — so 5% total, matching our p < 0.05 threshold. This is why large-sample statistics textbooks use 1.96 everywhere. In this course, we prefer RI because it works at any sample size and requires no distributional assumption. But for the power sweep below, running RI inside each of 1,000 fake experiments would take hours, so we use this shortcut. If you take a statistics course later, this is the *central limit theorem* in action.

In [ ]:
# Power at different sample sizes
# Using a normal approximation shortcut for speed here:
# significant if |ATE| > 1.96 * sqrt(2 * p * (1-p) / n_per_group)
# This gives the same answer as RI but runs much faster for large n.

np.random.seed(2026)
true_effect = 0.02
true_control_rate = 0.42
true_treatment_rate = 0.44

sample_sizes = [400, 1000, 2000, 5000, 10000, 20000]

print(f'{"n (total)":>10}  {"Power":>8}')
print('-' * 22)

for n_total in sample_sizes:
    n_per = n_total // 2
    # Approximate threshold for significance
    threshold = 1.96 * np.sqrt(2 * 0.43 * 0.57 / n_per)
    detected = 0
    for i in range(1000):
        control = np.random.binomial(1, true_control_rate, size=n_per)
        treatment = np.random.binomial(1, true_treatment_rate, size=n_per)
        ate = treatment.mean() - control.mean()
        if abs(ate) > threshold:
            detected += 1
    power = detected / 1000
    print(f'{n_total:>10,}  {power:>8.1%}')

**Read the table.** At n = 400, power is about 6%. At n = 10,000 it is still only about 50%. Power reaches about 80% at n = 20,000. You need roughly 20,000 households to reliably detect a 2-point canvassing effect.

This is the answer to the field director: the pilot was not wrong, it was just too small. To actually answer the question, you need an experiment 50 times larger.


## Part 4: Finding the minimum detectable effect (MDE)

The slides defined MDE as the smallest true effect that an experiment can reliably detect (at 80% power). We can find it by simulation: try different true effect sizes at the pilot's sample size (n = 400) and find where power reaches 80%.

In [ ]:
# MDE: try different true effect sizes at n = 400
# For each effect size, compute power. Find where power reaches 80%.

np.random.seed(2026)
n_per_group = 200
true_control_rate = 0.42

effect_sizes = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16]

print(f'{"True effect":>12}  {"Power":>8}')
print('-' * 24)

best_effect = None
best_dist = 1.0

for effect in effect_sizes:
    true_treatment_rate = true_control_rate + effect
    threshold = 1.96 * np.sqrt(2 * 0.43 * 0.57 / n_per_group)
    detected = 0
    for i in range(1000):
        control = np.random.binomial(1, true_control_rate, size=n_per_group)
        treatment = np.random.binomial(1, true_treatment_rate, size=n_per_group)
        ate = treatment.mean() - control.mean()
        if abs(ate) > threshold:
            detected += 1
    power = detected / 1000
    dist = abs(power - 0.80)
    if dist < best_dist:
        best_dist = dist
        best_effect = effect
    print(f'{effect:>11.0%}  {power:>8.1%}')

print(f'\nClosest to 80% power: {best_effect:.0%} true effect  --> this is the MDE')

The MDE for the pilot (n = 400) is roughly 14 percentage points. Power reaches 80% when the true effect is around 14 pp.

Published canvassing effects are +1 to +3 pp. The pilot could only reliably detect effects five to fourteen times that large. That is why the slides said "the pilot was never going to find them."

## Optional: visualize power vs. sample size


In [ ]:
import matplotlib.pyplot as plt

np.random.seed(2026)
sizes = list(range(400, 20001, 800))
powers = []
for n_total in sizes:
    n_per = n_total // 2
    threshold = 1.96 * np.sqrt(2 * 0.43 * 0.57 / n_per)
    detected = sum(1 for _ in range(500)
                   if abs(np.random.binomial(1, 0.44, n_per).mean()
                          - np.random.binomial(1, 0.42, n_per).mean()) > threshold)
    powers.append(detected / 500)

plt.figure(figsize=(8, 4))
plt.plot(sizes, powers, linewidth=2)
plt.axhline(0.80, color='red', linestyle='--', label='80% power')
plt.xlabel('Total sample size')
plt.ylabel('Power')
plt.title('Power to detect a +2 pp canvassing effect')
plt.legend()
plt.tight_layout()
plt.show()